## Encoding
### Opdracht

Zoals aan het begin van de cursus besproken gaat het project over handgeschreven nummers herekennen op een MysteryDevice met de volgende eigenschappen:

- Input scherm waarmee een nieuwe plaatje als ndarray aangemaakt kan worden door de gebruiker.
- Zeer beperkt RAM (256 KB)
- Beperkte opslag (1 MB voor 
programma + model)
- Geen GPU
- Embedded python


MysteryDevice opslaglimiet: 1 MB
Volledige MNIST dataset: 52 MB

Dus het opslaan van MNST voor training doeleinden zou niet werken. Maar in ons geval maakt het ook niet uit, want we kunnen onze AI methode op een PC trainen.

Echter, hebben we nog steeds een encoding van afbeeldingen nodig omdat de gebruiker wel een afbeelding invoert.

#### Hoe ziet de input eruit tijdens inference?

De gebruiker tekent op het touchscreen → dat moet naar 28×28 of een andere encoding worden gebracht.

#### Hoeveel RAM gebruikt die inputrepresentatie?

784 bytes (1 byte per pixel) -> heel laag
maar 32‑bits floats → 784 × 4 = 3 KB -> al een stuk hoger

Ontwerp een encoding van een MNIST plaatje die zo min mogelijk geheugen nodig heeft op het MysteryDevice om je Decision Tree uit de vorige opdracht te draaien op 1 sample. 

Mogelijk heb je al wat gedaan in de vorige stap, maar kijk of je het nog meer kan verbeteren.
Het moet:

- minder geheugen gebruiken
- minder rekenkracht nodig hebben
- toch informatie bewaren om cijfers te herkennen

Maak eventueel gebruik aan de volgende onderweren, maar ga vooral zelf experimenteren:
- binning
(“donker / medium / licht” → 3 waarden)

- binary thresholding
(zwart/wit → 1 bit per pixel)

- downscaling
(28×28 → 14×14 → 196 features)

- flattening
(784 → 784 vector)

- quantization
(8-bit → 4-bit → 2-bit per pixel)

- feature extraction
(bijv. “hoeveel inkt zit links/rechts/boven/onder?”)
- sparse encoding
(alleen niet‑nul pixels bewaren)

- Ga je normaliseren? Hoe?
- Ga je flattenen of 2D laten staan?
- Ga je binning toepassen op pixelwaarden?
- Ga je ordinale encoding toepassen op pixelintensiteiten?
- Ga je pixelwaarden reduceren (bijv. “donker, medium, licht”)?
- Ga je één pixel gebruiken, of alle 784?

**Ga ook een eigen “encoding” bedenken!**

### Mijn keuzes voor de encoding

Voordat ik begin met coderen, beantwoord ik eerst de ontwerpvragen:

- **Normaliseren?** Nee, niet in floats. Een Decision Tree heeft geen normalisatie nodig (boomsplitsingen werken op drempels, niet op schalen). Floats kosten juist 4× meer geheugen.
- **Flattenen of 2D?** Flattenen → een Decision Tree verwacht een 1D feature vector.
- **Binning?** Ja, ik gebruik **binary thresholding** (1 bit per pixel). MNIST cijfers zijn in essentie zwart/wit, dus grijswaarden voegen weinig toe.
- **Ordinale encoding?** Niet nodig na binary thresholding (er zijn maar 2 waarden).
- **Pixelwaarden reduceren?** Ja: 256 niveaus → 2 niveaus (zwart/wit).
- **Eén pixel of alle 784?** Ik gebruik **downscaling** van 28×28 → 14×14 = 196 pixels. Cijfers blijven goed herkenbaar bij die resolutie.

#### Mijn eigen encoding: "Downscaled Bit-packed Binary"

Ik combineer drie technieken:
1. **Downscaling** 28×28 → 14×14 (4× minder pixels)
2. **Binary thresholding** (alleen 0 of 1 per pixel)
3. **Bit-packing** (8 binaire pixels in 1 byte stoppen)

Resultaat: 196 bits = **25 bytes** per afbeelding (in plaats van 784 bytes of 3136 bytes voor floats).

In [ ]:
# Setup: MNIST sample laden om de encoding op te testen
import numpy as np
from sklearn.datasets import load_digits

# We gebruiken sklearn digits als snelle proxy voor MNIST (8x8), 
# maar we maken een 28x28 sample door op te schalen voor realistische test.
# In de praktijk komt 'sample' uit MNIST.
try:
    from sklearn.datasets import fetch_openml
    mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
    sample = mnist.data[0].reshape(28, 28).astype(np.uint8)
except Exception:
    # Fallback: maak een 28x28 sample uit sklearn digits
    digits = load_digits()
    small = digits.images[0].astype(np.uint8)  # 8x8
    # Schaal op naar 28x28 met simpele nearest-neighbor herhaling
    sample = np.kron(small, np.ones((4, 4), dtype=np.uint8))[:28, :28]
    sample = (sample * (255 // sample.max())).astype(np.uint8)

print("Sample shape:", sample.shape)
print("Sample dtype:", sample.dtype)
print("Sample nbytes (origineel):", sample.nbytes, "bytes")
sample

In [ ]:
# je kunt de volgende template gebruiken hier

def encode(img):
    # Stap 1: Downscaling 28x28 -> 14x14 door 2x2 blokken te middelen
    # (gebruikt alleen integer rekenwerk, geen floats nodig)
    small = (img[0::2, 0::2].astype(np.uint16)
             + img[1::2, 0::2]
             + img[0::2, 1::2]
             + img[1::2, 1::2]) // 4   # 14x14, waarden 0-255
    
    # Stap 2: Binary thresholding (drempel = 127)
    binary = (small > 127).astype(np.uint8)  # 14x14, waarden 0 of 1
    
    # Stap 3: Flatten naar 1D vector (196 bits)
    flat = binary.flatten()
    
    # Stap 4: Bit-packing: 8 pixels per byte -> 25 bytes
    encoded = np.packbits(flat)
    
    return encoded

encoded_sample = encode(sample)
encoded_sample

# bepaal hoeveel het gebruikt
encoded_sample.nbytes

### Controle: is het cijfer nog herkenbaar?

Laten we de originele en de gecodeerde (gedecodeerde) afbeelding naast elkaar tonen om te checken of er geen cruciale informatie verloren is gegaan.

In [ ]:
import matplotlib.pyplot as plt

def decode_for_view(encoded, shape=(14, 14)):
    """Pak de bits weer uit voor visuele controle."""
    bits = np.unpackbits(encoded)[:shape[0] * shape[1]]
    return bits.reshape(shape)

decoded = decode_for_view(encoded_sample)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(sample, cmap='gray')
axes[0].set_title(f'Origineel 28x28\n{sample.nbytes} bytes')
axes[0].axis('off')

axes[1].imshow(decoded, cmap='gray')
axes[1].set_title(f'Encoded 14x14 binary\n{encoded_sample.nbytes} bytes')
axes[1].axis('off')

plt.tight_layout()
plt.show()

Beantwoord ook de volgende vragen:

- Hoeveel RAM kost één afbeelding?
- Hoeveel RAM kost 100 afbeeldingen?
- Hoe groot zou het model maximaal mogen zijn?
- Kun je de encoding verder comprimeren?
- Is er informatie verloren gegaan?
- Kun je nog steeds het cijfer herkennen?

In [ ]:
# Berekeningen voor de antwoorden

een_afbeelding = encoded_sample.nbytes
honderd_afbeeldingen = een_afbeelding * 100

RAM_LIMIET = 256 * 1024       # 256 KB
OPSLAG_LIMIET = 1 * 1024 * 1024  # 1 MB

# Schatting programma-overhead
programma_overhead = 100 * 1024  # ~100 KB voor python runtime + code
max_model = OPSLAG_LIMIET - programma_overhead

print(f"1 afbeelding kost:        {een_afbeelding} bytes")
print(f"100 afbeeldingen kosten:  {honderd_afbeeldingen} bytes ({honderd_afbeeldingen/1024:.1f} KB)")
print(f"RAM limiet:               {RAM_LIMIET} bytes ({RAM_LIMIET/1024:.0f} KB)")
print(f"Opslag limiet:            {OPSLAG_LIMIET} bytes ({OPSLAG_LIMIET/1024:.0f} KB)")
print(f"Max modelgrootte (~):     {max_model} bytes ({max_model/1024:.0f} KB)")
print()
print(f"Vergelijking met andere encodings voor 1 afbeelding:")
print(f"  float32 28x28:          {28*28*4} bytes")
print(f"  uint8 28x28:            {28*28} bytes")
print(f"  uint8 14x14:            {14*14} bytes")
print(f"  binary 14x14 (packed):  {een_afbeelding} bytes  <-- mijn encoding")
print(f"  Compressiefactor t.o.v. float32: {28*28*4 / een_afbeelding:.0f}x kleiner")

### Antwoorden op de vragen

**Hoeveel RAM kost één afbeelding?**  
25 bytes met mijn encoding (downscaled 14×14 + binary + bit-packed). Tijdens het encoderen is er kort wat extra geheugen nodig voor de tussenstappen (~196 bytes voor de binary array vóór packing), maar het uiteindelijke object is 25 bytes.

**Hoeveel RAM kost 100 afbeeldingen?**  
2.500 bytes ≈ 2,5 KB. Past ruim in het 256 KB RAM-budget — er blijft ~253 KB over voor het model en de runtime.

**Hoe groot zou het model maximaal mogen zijn?**  
De totale opslag is 1 MB voor programma + model. Reken ~100 KB voor de Python runtime/code, dan blijft er ongeveer **900 KB** over voor het Decision Tree model. Een Decision Tree met een paar honderd nodes past daar makkelijk in (een node ≈ 10-20 bytes).

**Kun je de encoding verder comprimeren?**  
Ja, mogelijke verdere stappen:
- **Verder downscalen** naar 10×10 = 100 bits = 13 bytes (maar herkenbaarheid neemt af).
- **Feature extraction** in plaats van pixels: bijv. 16 features (inkt per kwadrant, symmetrie, aantal gaten) = 16 bytes. Dat is kleiner én sneller voor een Decision Tree.
- **Run-length encoding** of **sparse encoding** (alleen indices van zwarte pixels) — werkt goed omdat MNIST veel witte achtergrond heeft.

**Is er informatie verloren gegaan?**  
Ja, drie keer:
1. Downscaling 28×28 → 14×14 verliest fijne details.
2. Binary thresholding gooit alle grijswaarden weg.
3. De drempel van 127 is een vaste keuze — zachte randen verdwijnen.

**Kun je nog steeds het cijfer herkennen?**  
Ja, zoals te zien in de visualisatie hierboven. MNIST-cijfers zijn dik en goed leesbaar, ook bij 14×14 binary. Een Decision Tree heeft genoeg aan de globale vorm; pixelnauwkeurigheid is niet kritisch.

### Conclusie

| Eigenschap | Origineel (float32) | Mijn encoding |
|---|---|---|
| Bytes per afbeelding | 3.136 | **25** |
| Bytes per 100 afbeeldingen | 313.600 | **2.500** |
| Past in 256 KB RAM? | nauwelijks (1 plaatje al ~3 KB) | makkelijk (10.000+ plaatjes) |
| Cijfer herkenbaar? | ja | ja |

Met een compressie van ~125× ten opzichte van float32 past deze encoding ruim binnen de eisen van het MysteryDevice, en blijft de Decision Tree gewoon werken op de 196 binaire features.